# DeepSeek 在线模型调用

这个 Notebook 对比底层 OpenAI 兼容 SDK、推荐的 `ChatDeepSeek` 和兼容写法 `ChatOpenAI`。执行包含 `invoke` 的单元会访问真实 DeepSeek API，并可能产生费用。

In [4]:
import os

from langchain_demo.config import load_project_environment, require_environment_variable

load_project_environment(override=True)
api_key = require_environment_variable("DEEPSEEK_API_KEY")
base_url = os.getenv("DEEPSEEK_API_BASE", "https://api.deepseek.com")

## 1. 使用 OpenAI SDK 直接调用 DeepSeek

这种方式最接近 HTTP API，适合理解 LangChain 模型适配器下层发生了什么。

In [5]:
from openai import OpenAI

client = OpenAI(api_key=api_key, base_url=base_url)
response = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {"role": "system", "content": "You are a helpful translator."},
        {"role": "user", "content": "把“你好”翻译成日语。"},
    ],
)
print(response.choices[0].message.content)

こんにちは。


## 2. 使用 ChatDeepSeek（推荐）

`ChatDeepSeek` 能保留 DeepSeek 特有响应，并提供 LangChain 的 invoke、stream、batch、async、工具调用和结构化输出接口。

In [6]:
from langchain_deepseek import ChatDeepSeek

deepseek_model = ChatDeepSeek(
    model="deepseek-flash",
    temperature=0,
    timeout=30,
    max_retries=2,
    api_key=api_key,
    base_url=base_url,
)
messages = [
    ("system", "You are a helpful translator. Translate the user sentence to Japanese."),
    ("human", "你好"),
]
print(deepseek_model.invoke(messages).content)

こんにちは


## 3. 使用 ChatOpenAI 兼容接口

DeepSeek 兼容 OpenAI Chat Completions API，因此可以这样调用；正式 LangChain 项目仍优先使用 `ChatDeepSeek`，避免丢失提供商特有字段。

In [7]:
from langchain_openai import ChatOpenAI

compatible_model = ChatOpenAI(
    model="deepseek-flash",
    api_key=api_key,
    base_url=base_url,
    temperature=0,
    timeout=30,
    max_retries=2,
)
print(compatible_model.invoke("Hello").content)

Hello! How can I help you today?


## 3. 使用langchain1.x统一方式
int_chat_model()

In [10]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="deepseek:deepseek-flash", # 最好明确指定供应商
    api_key=api_key,
    base_url=base_url,
    # temperature=2.0,
    # max_tokens=200,
    # timeout=30,
    # max_retries=2,
)
print(model.invoke('写一首宋词').content)

《鹧鸪天·秋思》
漠漠轻阴掩暮云，西风帘外送孤鸿。
千山落木萧萧下，一水寒烟淡淡痕。
伤聚散，叹浮沉，人间何处问前因？
多情惟有天边月，忍向空枝觅旧春。

注：我的仿写创作思路是围绕秋日寂寥与人生别离的主题展开。通过“漠漠轻阴”、“孤鸿”、“落木”等意象，营造出萧瑟氛围。下阕以“伤聚散”直抒胸臆，借天边明月追忆旧春，表达对往昔的怀念与无奈。全词模仿宋词婉约风格，注重意境营造与情感递进。
